

# Multimodal RAG with Elasticsearch: The Gotham City Case



This notebook implements the Multimodal RAG (Retrieval-Augmented Generation) pipeline with Elasticsearch as described in the blog [Building a multimodal RAG system with Elasticsearch](https://www.elastic.co/search-labs/blog/building-multimodal-rag-system). We follow the same structure as the article, with each section explained and implemented in code.

## Environment Setup

First, we need to clone the repository that contains the complete project code.

```
!git clone https://github.com/elastic/elasticsearch-labs.git
```

Let's navigate to the project directory where the necessary files are located:
```
cd elasticsearch-labs/supporting-blog-content/building-multimodal-rag-with-elasticsearch-gotham
```

We recommend to create and start a virtual environment before running this tutorial:
```
python3 -m venv .venv
source .venv/bin/activate
```

Now let's configure the environment variables needed to connect to Elasticsearch and OpenAI-compatible APIs. This is necessary for indexing and searching content, as well as generating the final report.


In [ ]:
import getpass

ELASTICSEARCH_URL = input("Enter the Elasticsearch endpoint URL: ")
ELASTICSEARCH_API_KEY = getpass.getpass("Enter the Elasticsearch API key: ")
OPENAI_API_KEY = getpass.getpass("Enter the OpenAI or LiteLLM virtual API key: ")
OPENAI_BASE_URL = input("Enter LiteLLM base URL (leave empty for direct OpenAI): ").strip()

In [ ]:
import os

os.environ["ELASTICSEARCH_API_KEY"] = ELASTICSEARCH_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["ELASTICSEARCH_URL"] = ELASTICSEARCH_URL
if OPENAI_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL


## Installing Dependencies

As mentioned in the blog, we install the pinned dependencies used by this tutorial:


In [25]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Stage 1 - Collecting Crime Scene Clues

As explained in the blog, the first step is to verify that we have the correct directory structure and that the evidence files are present. We use `files_check.py` for this.

In [26]:
!python stages/01-stage/files_check.py

All files are correctly organized!


## Stage 2 - Generating Embeddings with EIS

Now we test embedding generation for an image. The `EmbeddingGenerator` automatically creates the EIS embedding endpoint for `jina-embeddings-v5-omni-small` if it does not exist.


In [27]:
!python stages/02-stage/test_embedding_generation.py

/Users/leonie/Documents/code/elasticsearch-labs/supporting-blog-content/building-multimodal-rag-with-elasticsearch-gotham/stages/02-stage/test_embedding_generation.py:18: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  text_response = es.inference.embedding(
Text embedding dims: 1024
/Users/leonie/Documents/code/elasticsearch-labs/supporting-blog-content/building-multimodal-rag-with-elasticsearch-gotham/stages/02-stage/test_embedding_generation.py:41: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  image_response = es.inference.embedding(
Image embedding dims: 1024
/Users/leonie/Documents/code/elast

This script generates a 1024-dimensional embedding for a test image, confirming that the EIS endpoint is working correctly.


In [28]:
# Delete the index if it exists
from elasticsearch import Elasticsearch
import os

es = Elasticsearch(
    os.getenv("ELASTICSEARCH_URL"),
    api_key=os.getenv("ELASTICSEARCH_API_KEY"),
)

index_name = "multimodal_content"

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Deleted index: {index_name}")
else:
    print(f"Index does not exist: {index_name}")

Index does not exist: multimodal_content



## Stage 3 - Storage and Search in Elasticsearch

### Content Indexing

The next step is to index all multimodal evidence in Elasticsearch. This includes images, audio, and text as described in the blog.

In [29]:
!python stages/03-stage/index_all_modalities.py

INFO:elastic_transport.transport:HEAD https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:404 duration:0.471s]
INFO:elastic_transport.transport:PUT https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:200 duration:1.326s]
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_doc [status:201 duration:3.643s]
INFO:__main__:

Indexed vision: {
  "result": "created",
  "_id": "rKwXO54BIj1pQ-jILRZK",
  "_index": "multimodal_content"
}
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_doc [status:201 duration:21.676s]
INFO:__main__:

Indexed vision: {
  "result": "created",
  "_id": "rawXO54BIj1pQ-jIOxb8",
  "_index": "multimodal_content"
}
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_doc [status:201 duration:25.


Each piece of evidence is now indexed in Elasticsearch with their respective embeddings, allowing for similarity search.

### Searching by Similarity Across Different Modalities

Now we can test searching for evidence by similarity using different modalities as queries. The blog describes how an input from one modality can retrieve results from all modalities.



#### Search by Text

Here we use a text query ("Why so serious?") to find related evidence.

In [30]:
!python stages/03-stage/search_by_text.py

INFO:elastic_transport.transport:HEAD https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:200 duration:0.476s]

🧾 Query used for search:
Why so serious?

INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:1.427s]

🔎 Similar evidence found:

1. The Joker with green hair, white face paint, and a sinister smile in an urban night setting. (vision)
   Similarity: 0.5831
   File path: data/images/joker_laughing.png

2. Suspect dancing (vision)
   Similarity: 0.5799
   File path: data/images/jdancing.png

3. Photo of the crime scene: A dark, rain-soaked alley is filled with playing cards, while a sinister graffiti of the Joker laughing stands out on the brick wall. (vision)
   Similarity: 0.5765
   File path: data/images/crime_scene1.jpg



#### Search by Audio

This command uses an audio file as a query and retrieves the most similar evidence. In the case of Gotham, this helps identify connections between the audio of a sinister laugh and other evidence.

In [31]:
!python stages/03-stage/search_by_audio.py

INFO:elastic_transport.transport:HEAD https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:200 duration:0.411s]

🧾 Query used for search:
data/audios/joker_laugh_similar.wav

INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:6.580s]

🔎 Similar evidence found:

1. A sinister laugh captured near the crime scene (audio)
   Similarity: 0.9098
   File path: data/audios/joker_laugh.wav

2. Suspect dancing (vision)
   Similarity: 0.8764
   File path: data/images/jdancing.png

3. Photo of the crime scene: A dark, rain-soaked alley is filled with playing cards, while a sinister graffiti of the Joker laughing stands out on the brick wall. (vision)
   Similarity: 0.8585
   File path: data/images/crime_scene1.jpg



#### Search by Image

This script uses an image from the crime scene to find similar visual evidence.

In [32]:
!python stages/03-stage/search_by_image.py

INFO:elastic_transport.transport:HEAD https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:200 duration:0.446s]

🧾 Query used for search:
data/images/crime_scene2.jpg

INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:5.337s]

🔎 Similar evidence found:

1. Photo of the crime scene: A dark, rain-soaked alley is filled with playing cards, while a sinister graffiti of the Joker laughing stands out on the brick wall. (vision)
   Similarity: 0.9583
   File path: data/images/crime_scene1.jpg

2. The Joker with green hair, white face paint, and a sinister smile in an urban night setting. (vision)
   Similarity: 0.9480
   File path: data/images/joker_laughing.png

3. Suspect dancing (vision)
   Similarity: 0.9235
   File path: data/images/jdancing.png



## Stage 4 - Evidence Analysis with LLM

Finally, we bring together all the retrieved evidence and use an LLM (GPT-4) to generate a forensic report that identifies the suspect based on the connections between the different modalities.


In [33]:
!python stages/04-stage/rag_crime_analyze.py

INFO:elastic_transport.transport:HEAD https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content [status:200 duration:0.434s]
INFO:__main__:✅ All components initialized successfully
INFO:__main__:🔍 Collecting evidence...
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:1.928s]
INFO:__main__:✅ Data retrieved for vision: 2 results
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:0.407s]
INFO:__main__:✅ Data retrieved for audio: 2 results
INFO:elastic_transport.transport:POST https://jina-test-b27719.es.us-central1.gcp.elastic.cloud:443/multimodal_content/_search [status:200 duration:1.171s]
INFO:__main__:✅ Data retrieved for text: 2 results
INFO:__main__:
📝 Generating forensic report...
INFO:httpx:HTTP Request: POST https://elastic.litellm-prod.ai/chat/completions "HTTP/1


This is the final step of the Multimodal RAG pipeline, where the LLM analyzes the evidence retrieved from Elasticsearch and synthesizes it into a coherent report that identifies the Joker as the main suspect.

## Conclusion

We have thus completed the implementation of the complete Multimodal RAG pipeline with Elasticsearch, following all the steps described in the blog. This pipeline demonstrates how different types of media can be analyzed in an integrated way to provide richer insights and connections between evidence that would be difficult to identify manually.
